<h2 align="center" style="color:#4169E1;">
Hyper parameter Tuning </h2>

In [1]:
import os
from torchvision import datasets,transforms
import torch 
import torch.nn as nn 
import torch.optim as optim
import torch.nn.functional as F 
import torchvision.models as models
from torch.utils.data import DataLoader,random_split
import time
import matplotlib.pyplot as plt

In [2]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

## Load Data

In [3]:
img_transforms=transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(
                                mean = [0.485, 0.456, 0.406],
                                std  = [0.229, 0.224, 0.225]
                            )
    ]
)

In [4]:
path=".\dataset"
dataset=datasets.ImageFolder(root=path,transform=img_transforms)
len(dataset)

2300

In [5]:
classes=dataset.classes
classes

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [6]:
num_classes=len(classes)
num_classes

6

In [7]:
train_size=int(0.75*len(dataset))
val_size=len(dataset)-train_size

train_size,val_size

(1725, 575)

In [8]:
train_dataset,val_dataset=random_split(dataset,[train_size,val_size])

In [9]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=True)


In [10]:
class CarClassifierCNNResNet(nn.Module):
    def __init__(self,num_classes,dropout=0.5):
        super().__init__()
        self.model=models.resnet50(weights="DEFAULT")

        #freeze Before Layers
        for params in self.model.parameters():
            params.requires_grad=False
        # Unfreeze layer 4
        for params in self.model.layer4.parameters():
            params.requires_grad=True

        # FC Replacement
        self.model.fc=nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.model.fc.in_features,num_classes)
        )
    
    def forward(self,x):
        return self.model(x)

In [12]:
import optuna

In [ ]:
# Define the objective function for Optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    
    # Load the model
    model = CarClassifierCNNResNet(num_classes=num_classes, dropout=dropout_rate).to(device)
    
    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    # Training loop (using fewer epochs for faster hyperparameter tuning)
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total   
        
        # Report intermediate result to Optuna
        trial.report(accuracy, epoch)
        
        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time: {end - start} seconds")
    
    return accuracy

In [ ]:
# Create the study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-02-23 16:04:10,112] A new study created in memory with name: no-name-d601fbe0-2eb9-4e6e-a02f-b60178a485b4
[I 2026-02-23 16:07:33,101] Trial 0 finished with value: 77.56521739130434 and parameters: {'lr': 0.000609271560579746, 'dropout_rate': 0.6967788350946911}. Best is trial 0 with value: 77.56521739130434.


Execution time: 200.69692730903625 seconds


[I 2026-02-23 16:10:29,337] Trial 1 finished with value: 76.17391304347827 and parameters: {'lr': 0.0019074034205840053, 'dropout_rate': 0.29354658141811096}. Best is trial 0 with value: 77.56521739130434.


Execution time: 175.9417941570282 seconds


[I 2026-02-23 16:13:26,970] Trial 2 finished with value: 71.30434782608695 and parameters: {'lr': 5.459909888062072e-05, 'dropout_rate': 0.46756915980327785}. Best is trial 0 with value: 77.56521739130434.


...

Execution time: 284.2618989944458 seconds


[I 2026-02-23 16:22:04,791] Trial 5 pruned. 
[I 2026-02-23 16:25:11,507] Trial 6 finished with value: 77.21739130434783 and parameters: {'lr': 0.0001026400382429596, 'dropout_rate': 0.20311713476298932}. Best is trial 0 with value: 77.56521739130434.


Execution time: 186.41667652130127 seconds


[I 2026-02-23 16:30:07,286] Trial 7 pruned. 
[I 2026-02-23 16:31:05,765] Trial 8 pruned. 
[I 2026-02-23 16:34:03,595] Trial 9 pruned. 
[I 2026-02-23 16:36:01,228] Trial 10 pruned. 
[I 2026-02-23 16:38:58,473] Trial 11 finished with value: 77.56521739130434 and parameters: {'lr': 0.0005501659801460218, 'dropout_rate': 0.3651653278030736}. Best is trial 0 with value: 77.56521739130434.


Execution time: 176.99376845359802 seconds


In [ ]:
study.best_params